# 03 Ensemble Prediction and Test-Time Augmentation

This notebook combines the saved ConvNeXt Tiny and EfficientNetV2-S checkpoints from Notebooks 02a–02c. It compares weighted probability ensembles with horizontal-flip test-time augmentation, examines remaining validation errors, and explores image-shape adjustments estimated from the training split.

## 1 Setup, checkpoints and validation split

The three checkpoints from Notebooks 02a–02c are located, and the original stratified validation split is reconstructed using the same cleaned metadata and random seed. Each model retains its original input resolution.

In [1]:
import os, glob, random
import numpy as np
import pandas as pd
import torch, timm
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Same seed as notebook 02 -> identical validation split
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# Locate the competition data automatically (works regardless of the exact input path)
sub_paths = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
print("sample_submission found at:", sub_paths)
BASE = os.path.dirname(sub_paths[0])
TRAIN_DIR = os.path.join(BASE, "train", "train")
TEST_DIR  = os.path.join(BASE, "test", "test")

meta_path = glob.glob("/kaggle/input/**/train_metadata_clean.csv", recursive=True)
print("Metadata:", meta_path)

# Model registry: checkpoint name -> (timm architecture, input size)
MODEL_REGISTRY = {
    "convnext_t_128": ("convnext_tiny", 128),
    "convnext_t_224": ("convnext_tiny", 224),
    "effnetv2_s_128": ("tf_efficientnetv2_s", 128),
}

# Find all checkpoints once, then match them to the registry by file name
all_ckpts = glob.glob("/kaggle/input/**/best_*.pth", recursive=True)
ckpt_paths = {}
for key in MODEL_REGISTRY:
    matches = [p for p in all_ckpts if os.path.basename(p) == f"best_{key}.pth"]
    assert matches, f"Checkpoint for {key} not found!"
    ckpt_paths[key] = matches[0]
for key, path in ckpt_paths.items():
    print(f"{key:16s} -> {path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Rebuild labels and the SAME stratified split as notebook 02
df = pd.read_csv(meta_path[0])
CLASSES = sorted(df["class"].unique())
class_to_idx = {c: i for i, c in enumerate(CLASSES)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
df["label"] = df["class"].map(class_to_idx)

_, val_df = train_test_split(df, test_size=0.15, stratify=df["label"], random_state=SEED)
val_df = val_df.reset_index(drop=True)
test_df = pd.read_csv(os.path.join(BASE, "sample_submission.csv"))[["image_id"]]

print(f"Val: {len(val_df)} | Test: {len(test_df)}")

sample_submission found at: ['/kaggle/input/datasets/khlafalshammari/competition/sample_submission.csv']
Metadata: ['/kaggle/input/notebooks/khlafalshammari/01-eda-facial-features/train_metadata_clean.csv']
convnext_t_128   -> /kaggle/input/datasets/khlafalshammari/eng707-models/best_convnext_t_128.pth
convnext_t_224   -> /kaggle/input/datasets/khlafalshammari/eng707-models/best_convnext_t_224.pth
effnetv2_s_128   -> /kaggle/input/datasets/khlafalshammari/eng707-models/best_effnetv2_s_128.pth
Device: cuda
Val: 9547 | Test: 29443


## 2 Evaluation preprocessing and data loading

In [2]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def make_eval_tf(size):
    # Same deterministic preprocessing as validation/test in notebook 02
    return T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(MEAN, STD)])

class FacialFeatureDataset(Dataset):
    def __init__(self, df, img_dir, transform, has_labels=True):
        self.ids = df["image_id"].values
        self.labels = df["label"].values if has_labels else None
        self.img_dir, self.transform = img_dir, transform
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, idx):
        img_id = int(self.ids[idx])
        img = Image.open(os.path.join(self.img_dir, f"{img_id:05d}.jpg")).convert("RGB")
        img = self.transform(img)
        return (img, int(self.labels[idx])) if self.labels is not None else (img, img_id)

def make_loader(df, img_dir, size, has_labels):
    ds = FacialFeatureDataset(df, img_dir, make_eval_tf(size), has_labels)
    return DataLoader(ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

## 3 Prediction, flip averaging and ensemble comparison

Each model predicts class probabilities for the original images and their horizontal flips. These probabilities are averaged within each model, then combined using several predefined ensemble weights. Validation accuracy, error count and log loss are compared.


In [3]:
import torch.nn.functional as F
from sklearn.metrics import log_loss

@torch.no_grad()
def predict_probs(model, loader):
    """Returns class probabilities for the original images AND their horizontal flips (TTA)."""
    model.eval()
    p_orig, p_flip = [], []
    for x, _ in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x)
            logits_flip = model(torch.flip(x, dims=[3]))   # flip left-right
        p_orig.append(F.softmax(logits.float(), dim=1).cpu())
        p_flip.append(F.softmax(logits_flip.float(), dim=1).cpu())
    return torch.cat(p_orig).numpy(), torch.cat(p_flip).numpy()

# Predict with every model (validation + test, original + flipped)
val_probs, test_probs = {}, {}
for key, (arch, size) in MODEL_REGISTRY.items():
    model = timm.create_model(arch, pretrained=False, num_classes=len(CLASSES))
    model.load_state_dict(torch.load(ckpt_paths[key], map_location=device))
    model.to(device)
    print(f"{key}: predicting validation and test...")
    val_probs[key]  = predict_probs(model, make_loader(val_df,  TRAIN_DIR, size, True))
    test_probs[key] = predict_probs(model, make_loader(test_df, TEST_DIR,  size, False))
    np.save(f"/kaggle/working/test_probs_{key}.npy", np.stack(test_probs[key]))
    del model; torch.cuda.empty_cache()

def tta(p):
    """Average of original and flipped predictions."""
    return (p[0] + p[1]) / 2

def blend(probs, weights):
    """Weighted average of TTA probabilities across models."""
    total = sum(weights.values())
    return sum(w * tta(probs[k]) for k, w in weights.items()) / total

# Candidate setups (all with flip TTA)
SETUPS = {
    "ConvNeXt 128":                   {"convnext_t_128": 1},
    "ConvNeXt 224":                   {"convnext_t_224": 1},
    "EffNetV2-S 128":                 {"effnetv2_s_128": 1},
    "ConvNeXt 128+224 (Kaggle 0.9783)": {"convnext_t_128": 1, "convnext_t_224": 1},
    "All 3, equal weights":           {"convnext_t_128": 1, "convnext_t_224": 1, "effnetv2_s_128": 1},
    "All 3, EffNet half weight":      {"convnext_t_128": 1, "convnext_t_224": 1, "effnetv2_s_128": 0.5},
}

y_val = val_df["label"].values
rows = []
for name, w in SETUPS.items():
    p = blend(val_probs, w)
    preds = p.argmax(1)
    rows.append({"setup": name,
                 "val_acc": (preds == y_val).mean(),
                 "errors": int((preds != y_val).sum()),
                 "val_logloss": log_loss(y_val, p, labels=range(len(CLASSES)))})
comparison = pd.DataFrame(rows).set_index("setup").round(5)
comparison.to_csv("/kaggle/working/ensemble_comparison_3models.csv")
comparison

convnext_t_128: predicting validation and test...


  0%|          | 0/75 [00:00<?, ?it/s]

  0%|          | 0/231 [00:00<?, ?it/s]

convnext_t_224: predicting validation and test...


  0%|          | 0/75 [00:00<?, ?it/s]

...

effnetv2_s_128: predicting validation and test...


  0%|          | 0/75 [00:00<?, ?it/s]

  0%|          | 0/231 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dd39f8b6c00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    Exception ignored in: self._shutdown_workers()<function _MultiProcessingDataLoaderIter.__del__ at 0x7dd39f8b6c00>
Exception ignored in: 
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
<function _MultiProcessingDataLoaderIter.__del__ at 0x7dd39f8b6c00>Traceback (most recent call last):
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__

if w.is_alive():
Traceback (most recent call last):
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
self._shutdown_workers()        
self._shutdown_workers()assert self._

,val_acc,errors,val_logloss
setup,,,
ConvNeXt 128,0.99675,31,0.10004
ConvNeXt 224,0.99644,34,0.10163
EffNetV2-S 128,0.99487,49,0.11185
ConvNeXt 128+224 (Kaggle 0.9783),0.99717,27,0.09982
"All 3, equal weights",0.99738,25,0.10220
"All 3, EffNet half weight",0.99749,24,0.10100


## 4 Ensemble submission export

In [4]:
def save_submission(probs, name):
    sub = pd.read_csv(os.path.join(BASE, "sample_submission.csv"))
    sub["class"] = [idx_to_class[i] for i in probs.argmax(1)]
    assert len(sub) == 29443 and sub["image_id"].is_unique
    assert set(sub["class"]).issubset(set(CLASSES))
    path = f"/kaggle/working/submission_{name}.csv"
    sub.to_csv(path, index=False)
    print(f"Saved {path}")
    print(sub["class"].value_counts().to_dict(), "\n")

save_submission(blend(test_probs, SETUPS["All 3, equal weights"]),      "ensemble3_equal_tta")
save_submission(blend(test_probs, SETUPS["All 3, EffNet half weight"]), "ensemble3_effhalf_tta")
save_submission(blend(test_probs, SETUPS["EffNetV2-S 128"]),            "effnetv2_s_128_tta")

Saved /kaggle/working/submission_ensemble3_equal_tta.csv
{'CatEye': 6319, 'DogEye': 5570, 'CatMouth': 3263, 'CatFace': 3115, 'CatNose': 2881, 'DogMouth': 2809, 'DogFace': 2806, 'DogNose': 2680} 

Saved /kaggle/working/submission_ensemble3_effhalf_tta.csv
{'CatEye': 6319, 'DogEye': 5574, 'CatMouth': 3263, 'CatFace': 3116, 'CatNose': 2874, 'DogMouth': 2806, 'DogFace': 2806, 'DogNose': 2685} 

Saved /kaggle/working/submission_effnetv2_s_128_tta.csv
{'CatEye': 6363, 'DogEye': 5528, 'CatMouth': 3266, 'CatFace': 3130, 'CatNose': 2884, 'DogMouth': 2820, 'DogFace': 2804, 'DogNose': 2648} 



## 5 Error analysis by image size and shape

Remaining ensemble errors are examined alongside image dimensions, aspect ratios and class labels. Error rates are also grouped by image size to investigate whether small crops present greater difficulty.

In [5]:
# Cell 5: do image size / shape explain the remaining errors?
best_val = blend(val_probs, SETUPS["All 3, EffNet half weight"])
val_df["pred"] = best_val.argmax(1)
val_df["pred_class"] = val_df["pred"].map(idx_to_class)
val_df["correct"] = val_df["pred"] == val_df["label"]
val_df["square"] = (val_df["aspect"] - 1).abs() < 0.05      # aspect ratio ~1.0
val_df["short_side"] = val_df[["width", "height"]].min(axis=1)

# 1. How "square" is each class? (mouths should rarely be square)
print("Share of SQUARE images per class:")
print(val_df.groupby("class")["square"].mean().round(3), "\n")

# 2. The errors, with their shape and size
errs = val_df[~val_df["correct"]].sort_values(["class", "pred_class"])
print(f"{len(errs)} errors:")
print(errs[["image_id", "class", "pred_class", "width", "height", "aspect", "short_side"]].to_string(index=False))

# 3. Are errors concentrated in small images?
print("\nError rate by image size (shortest side):")
bins = pd.cut(val_df["short_side"], [0, 32, 64, 128, 10000], labels=["<32", "32-63", "64-127", "128+"])
print(val_df.groupby(bins, observed=True)["correct"].agg(images="size", error_rate=lambda s: round(1 - s.mean(), 4)))

Share of SQUARE images per class:
class
CatEye      1.000
CatFace     0.097
CatMouth    0.000
CatNose     0.994
DogEye      0.999
DogFace     0.968
DogMouth    0.000
DogNose     0.998
Name: square, dtype: float64 

24 errors:
 image_id    class pred_class  width  height  aspect  short_side
    11675   CatEye    CatNose    104     104    1.00         104
    30859   CatEye    CatNose     16      16    1.00          16
    31318   CatEye     DogEye     18      18    1.00          18
    53571   CatEye     DogEye     76      76    1.00          76
    51193   CatEye     DogEye     64      64    1.00          64
    30269   CatEye   DogMouth    178     178    1.00         178
    26664 CatMouth    CatNose     40      28    1.43          28
    56081  CatNose     CatEye     76      76    1.00          76
    24149  CatNose   CatMouth     72      72    1.00          72
    59309  CatNose   CatMouth    158     115    1.37         115
    54874  CatNose   CatMouth     52      52    1.00       

## 6 Square-shape probability adjustment

Class-specific probabilities of near-square images are estimated from the training portion only, with Laplace smoothing. These shape likelihoods are multiplied by the model probabilities and renormalised as an exploratory adjustment. Because the model may already use shape information, this can count the same evidence twice.

In [6]:
from sklearn.metrics import log_loss

# Estimate P(square | class) from the TRAINING portion only (no validation leakage)
train_part = df[~df["image_id"].isin(val_df["image_id"])].copy()
train_part["square"] = (train_part["aspect"] - 1).abs() < 0.05
counts = train_part.groupby("class")["square"].agg(["sum", "size"]).reindex(CLASSES)
p_square = ((counts["sum"] + 1) / (counts["size"] + 2)).values      # Laplace smoothing: never exactly 0 or 1
print(pd.DataFrame({"P(square | class)": p_square.round(4)}, index=CLASSES), "\n")

def shape_adjust(probs, is_square):
    """Bayes update: multiply model probabilities by P(observed shape | class), then renormalise."""
    likelihood = np.where(is_square[:, None], p_square[None, :], 1 - p_square[None, :])
    adjusted = probs * likelihood
    return adjusted / adjusted.sum(axis=1, keepdims=True)

y_val = val_df["label"].values
val_square = val_df["square"].values
rows = []
for name in ["ConvNeXt 128", "ConvNeXt 128+224 (Kaggle 0.9783)", "All 3, EffNet half weight"]:
    p = blend(val_probs, SETUPS[name])
    for label, probs in [("model only", p), ("+ shape", shape_adjust(p, val_square))]:
        preds = probs.argmax(1)
        rows.append({"setup": f"{name} | {label}",
                     "errors": int((preds != y_val).sum()),
                     "val_acc": round((preds == y_val).mean(), 5),
                     "val_logloss": round(log_loss(y_val, probs, labels=range(len(CLASSES))), 5)})
shape_results = pd.DataFrame(rows).set_index("setup")
shape_results.to_csv("/kaggle/working/shape_adjust_results.csv")
shape_results

          P(square | class)
CatEye               0.9993
CatFace              0.1083
CatMouth             0.0002
CatNose              0.9907
DogEye               0.9998
DogFace              0.9732
DogMouth             0.0004
DogNose              0.9996 



,errors,val_acc,val_logloss
setup,,,
ConvNeXt 128 | model only,31,0.99675,0.10004
ConvNeXt 128 | + shape,26,0.99728,0.06185
ConvNeXt 128+224 (Kaggle 0.9783) | model only,27,0.99717,0.09982
ConvNeXt 128+224 (Kaggle 0.9783) | + shape,26,0.99728,0.06194
"All 3, EffNet half weight | model only",24,0.99749,0.10100
"All 3, EffNet half weight | + shape",24,0.99749,0.06247


## 7 Corrected predictions and new errors
Predictions before and after shape adjustment are compared to distinguish corrected errors from newly introduced mistakes. This shows the trade-off behind any change in overall validation accuracy.

In [8]:
name = "All 3, EffNet half weight"
p_before = blend(val_probs, SETUPS[name])
p_after  = shape_adjust(p_before, val_square)
pred_before, pred_after = p_before.argmax(1), p_after.argmax(1)

view = val_df[["image_id", "class", "width", "height", "aspect", "square"]].copy()
view["pred_before"] = [idx_to_class[i] for i in pred_before]
view["pred_after"]  = [idx_to_class[i] for i in pred_after]

fixed  = view[(pred_before != y_val) & (pred_after == y_val)]
broken = view[(pred_before == y_val) & (pred_after != y_val)]
still  = view[(pred_before != y_val) & (pred_after != y_val)]

print(f"FIXED by shape ({len(fixed)}):");   print(fixed.to_string(index=False), "\n")
print(f"BROKEN by shape ({len(broken)}):"); print(broken.to_string(index=False), "\n")
print(f"STILL WRONG ({len(still)}):");      print(still.to_string(index=False))

FIXED by shape (11):
 image_id    class  width  height  aspect  square pred_before pred_after
    58867   DogEye     36      36    1.00    True    CatMouth     DogEye
    24149  CatNose     72      72    1.00    True    CatMouth    CatNose
    10353  CatNose    132     132    1.00    True    CatMouth    CatNose
    39416  CatNose     50      50    1.00    True    CatMouth    CatNose
    27310  CatNose    172     172    1.00    True    CatMouth    CatNose
    28945  CatNose    186     186    1.00    True    CatMouth    CatNose
    11423  CatNose     26      26    1.00    True    CatMouth    CatNose
    35930  CatNose     54      54    1.00    True    CatMouth    CatNose
    30292  CatNose     64      64    1.00    True    CatMouth    CatNose
    26664 CatMouth     40      28    1.43   False     CatNose   CatMouth
    26960  CatNose     24      24    1.00    True    CatMouth    CatNose 

BROKEN by shape (11):
 image_id   class  width  height  aspect  square pred_before pred_after
    173

## 8 Detailed aspect-ratio adjustment

A more detailed adjustment estimates class-specific likelihoods across several aspect-ratio bins using training images only. Validation results assess whether this additional shape information improves predictions or introduces new errors.

In [9]:
# 1. Look at the actual aspect-ratio distribution per class (training portion only)
print("Aspect ratio quantiles per class (training portion):")
print(train_part.groupby("class")["aspect"].quantile([0, 0.01, 0.05, 0.5, 0.95, 0.99, 1]).unstack().round(2), "\n")

# 2. Bin the aspect ratio and estimate P(bin | class) from training counts
EDGES = [0, 0.8, 0.9, 0.95, 1.05, 1.1, 1.2, 1.3, 1.4, 1.6, np.inf]
N_BINS = len(EDGES) - 1

def aspect_bin(aspects):
    return np.digitize(aspects, EDGES[1:-1])     # bin index 0 .. N_BINS-1

counts = np.zeros((N_BINS, len(CLASSES)))
for b, c in zip(aspect_bin(train_part["aspect"].values), train_part["class"].map(class_to_idx)):
    counts[b, c] += 1
lik = (counts + 1) / (counts.sum(axis=0, keepdims=True) + N_BINS)   # Laplace-smoothed P(bin | class)

bin_names = [f"{EDGES[i]}-{EDGES[i+1]}" for i in range(N_BINS)]
print("Training counts per aspect bin:")
print(pd.DataFrame(counts.astype(int), index=bin_names, columns=CLASSES), "\n")

def shape_adjust_bins(probs, aspects):
    adjusted = probs * lik[aspect_bin(aspects)]
    return adjusted / adjusted.sum(axis=1, keepdims=True)

# 3. Evaluate on validation
val_aspect = val_df["aspect"].values
rows = []
for name in ["ConvNeXt 128", "ConvNeXt 128+224 (Kaggle 0.9783)", "All 3, EffNet half weight"]:
    p = blend(val_probs, SETUPS[name])
    p_adj = shape_adjust_bins(p, val_aspect)
    before, after = p.argmax(1), p_adj.argmax(1)
    rows.append({"setup": name,
                 "errors_before": int((before != y_val).sum()),
                 "errors_after": int((after != y_val).sum()),
                 "fixed": int(((before != y_val) & (after == y_val)).sum()),
                 "broken": int(((before == y_val) & (after != y_val)).sum()),
                 "logloss_after": round(log_loss(y_val, p_adj, labels=range(len(CLASSES))), 5)})
binned_results = pd.DataFrame(rows).set_index("setup")
binned_results.to_csv("/kaggle/working/shape_adjust_binned_results.csv")
binned_results

Aspect ratio quantiles per class (training portion):
          0.00  0.01  0.05  0.50  0.95  0.99  1.00
class                                             
CatEye    0.93  1.00  1.00  1.00  1.00  1.00  1.67
CatFace   0.40  0.55  0.64  0.83  1.16  1.46  4.31
CatMouth  1.10  1.38  1.40  1.42  1.45  1.57  8.40
CatNose   0.96  1.00  1.00  1.00  1.00  1.02  3.93
DogEye    0.84  1.00  1.00  1.00  1.00  1.00  1.02
DogFace   0.83  0.95  0.99  1.00  1.01  1.06  1.22
DogMouth  1.00  1.31  1.33  1.35  1.37  1.38  2.03
DogNose   0.97  1.00  1.00  1.00  1.00  1.00  1.20 

Training counts per aspect bin:
           CatEye  CatFace  CatMouth  CatNose  DogEye  DogFace  DogMouth  \
0-0.8           0     2327         0        0       0        0         0   
0.8-0.9         0     1640         0        0       1        8         0   
0.9-0.95        2      533         0        0       0       36         0   
0.95-1.05   11788      714         0     5846    9839     4803         1   
1.05-1.1        1      

,errors_before,errors_after,fixed,broken,logloss_after
setup,,,,,
ConvNeXt 128,31,43,13,25,0.05270
ConvNeXt 128+224 (Kaggle 0.9783),27,42,11,26,0.05279
"All 3, EffNet half weight",24,39,11,26,0.05309


## 9 Coarse shape adjustment and strength comparison

Aspect ratios are grouped into tall, approximately square and wide categories. Several adjustment strengths are compared, with smaller values reducing the influence of shape. Selection among these alternatives uses the same validation set, so the results remain exploratory.

In [10]:
# Coarse shape bins: tall / square-ish / wide
EDGES_C = [0, 0.9, 1.25, np.inf]
SHAPES  = ["tall (<0.9)", "square-ish (0.9-1.25)", "wide (>1.25)"]

def coarse_bin(aspects):
    return np.digitize(aspects, EDGES_C[1:-1])

counts_c = np.zeros((len(SHAPES), len(CLASSES)))
for b, c in zip(coarse_bin(train_part["aspect"].values), train_part["class"].map(class_to_idx)):
    counts_c[b, c] += 1
lik_c = (counts_c + 1) / (counts_c.sum(axis=0, keepdims=True) + len(SHAPES))   # P(shape | class)

print("Training counts per coarse shape:")
print(pd.DataFrame(counts_c.astype(int), index=SHAPES, columns=CLASSES), "\n")

def shape_adjust_coarse(probs, aspects, alpha):
    """Tempered Bayes update: alpha < 1 softens the shape evidence."""
    adjusted = probs * lik_c[coarse_bin(aspects)] ** alpha
    return adjusted / adjusted.sum(axis=1, keepdims=True)

rows = []
for name in ["ConvNeXt 128+224 (Kaggle 0.9783)", "All 3, EffNet half weight"]:
    p = blend(val_probs, SETUPS[name])
    before = p.argmax(1)
    for alpha in [0.25, 0.5, 0.75, 1.0]:
        p_adj = shape_adjust_coarse(p, val_aspect, alpha)
        after = p_adj.argmax(1)
        rows.append({"setup": name, "alpha": alpha,
                     "errors_before": int((before != y_val).sum()),
                     "errors_after": int((after != y_val).sum()),
                     "fixed": int(((before != y_val) & (after == y_val)).sum()),
                     "broken": int(((before == y_val) & (after != y_val)).sum()),
                     "logloss_after": round(log_loss(y_val, p_adj, labels=range(len(CLASSES))), 5)})
coarse_results = pd.DataFrame(rows).set_index(["setup", "alpha"])
coarse_results.to_csv("/kaggle/working/shape_adjust_coarse_results.csv")
coarse_results

Training counts per coarse shape:
                       CatEye  CatFace  CatMouth  CatNose  DogEye  DogFace  \
tall (<0.9)                 0     3967         0        0       1        8   
square-ish (0.9-1.25)   11792     1740         3     5886    9839     4912   
wide (>1.25)                3      194      5897       14       0        0   

                       DogMouth  DogNose  
tall (<0.9)                   0        0  
square-ish (0.9-1.25)         9     4921  
wide (>1.25)               4912        0   



errors_before  errors_after  fixed  \
setup                            alpha                                       
ConvNeXt 128+224 (Kaggle 0.9783) 0.25              27            20      8   
                                 0.50              27            20      9   
                                 0.75              27            22     10   
                                 1.00              27            22     10   
All 3, EffNet half weight        0.25              24            18      8   
                                 0.50              24            16     10   
                                 0.75              24            18     11   
                                 1.00              24            18     11   

                                        broken  logloss_after  
setup                            alpha                         
ConvNeXt 128+224 (Kaggle 0.9783) 0.25        1        0.06299  
                                 0.50        2        0.05614  
                                 0.75        5        0.05579  
                                 1.00        5        0.05780  
All 3, EffNet half weight        0.25        2        0.06344  
                                 0.50        2        0.05651  
                                 0.75        5        0.05616  
                                 1.00        5        0.05819

## 10 Shape-adjusted test predictions and export

The coarse shape adjustment with strength 0.5 is applied to test probabilities using image dimensions only. Changed predictions are summarised and a candidate submission is exported. These changes are not known corrections because test labels are unavailable.

In [11]:
# Test image aspect ratios (reads only the image headers: ~2-3 minutes)
test_sizes = [Image.open(os.path.join(TEST_DIR, f"{int(i):05d}.jpg")).size for i in tqdm(test_df["image_id"])]
test_aspect = np.array([round(w / h, 2) for w, h in test_sizes])      # same rounding as notebook 01

shape_counts = pd.Series(coarse_bin(test_aspect)).map(dict(enumerate(SHAPES))).value_counts()
print("Test images per coarse shape:\n", shape_counts, "\n")

ALPHA = 0.5
p_test = blend(test_probs, SETUPS["All 3, EffNet half weight"])
p_test_shape = shape_adjust_coarse(p_test, test_aspect, ALPHA)

changed = p_test.argmax(1) != p_test_shape.argmax(1)
print(f"Test predictions changed by the shape correction: {changed.sum()}")
change_table = pd.crosstab(
    pd.Series([idx_to_class[i] for i in p_test.argmax(1)[changed]], name="before"),
    pd.Series([idx_to_class[i] for i in p_test_shape.argmax(1)[changed]], name="after"))
print(change_table, "\n")

save_submission(p_test_shape, "ensemble3_effhalf_tta_shape_a05")

  0%|          | 0/29443 [00:00<?, ?it/s]

Test images per coarse shape:
 square-ish (0.9-1.25)    17928
wide (>1.25)              6707
tall (<0.9)               4808
Name: count, dtype: int64 

Test predictions changed by the shape correction: 2222
after     CatEye  CatFace  CatMouth  CatNose  DogEye  DogFace  DogMouth  \
before                                                                    
CatEye         0      902        57        0       0        0        19   
CatFace        1        0         3        3       2        0         0   
CatMouth       6        3         0      122       2        1         0   
CatNose        0      164        68        0       0        0         0   
DogEye         0      109       190        0       0        0       243   
DogFace        0        3         2        0       0        0        19   
DogMouth       6        1         0        2       7        2         0   
DogNose        0       48         6        0       0        0       158   

after     DogNose  
before             
Ca

## Summary

The weighted ensemble combined ConvNeXt Tiny at 128 and 224 pixels with EfficientNetV2-S at 128 pixels, using weights of 40%, 40% and 20% respectively and horizontal-flip averaging. It achieved 99.7486% validation accuracy (24 errors across 9,547 images). The corresponding submission, `submission_ensemble3_effhalf_tta.csv`, achieved a Kaggle public score of 0.9785.

Exploratory coarse shape adjustment reduced validation errors to 16 at an adjustment strength of 0.5. This result came from comparing several alternatives on the same validation split and does not establish an improvement on the hidden test set.